# Model Training on v2 Dataset

This notebook trains/evaluates a model on `micro_mobility_training_data_2025_v2.csv`, reports training time and train/test metrics, saves artifacts for later inference/reporting, and supports loading artifacts without retraining.


## RandomForest Regressor


In [ ]:
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error
from google.colab import drive

drive.mount('/content/drive')

project_folder = '/content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction'
data_path = os.path.join(project_folder, 'data/proceed/micro_mobility_training_data_2025_v2.csv')

print('data_path:', data_path)


In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor


In [ ]:
# Load dataset and split chronologically (last 7 days test)
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
if 'datetime_hour' in df.columns:
    df['datetime_hour'] = pd.to_datetime(df['datetime_hour'])

if 'station' in df.columns and 'station_id' not in df.columns:
    df['station_id'] = pd.factorize(df['station'])[0].astype('int32')

sort_cols = ['datetime_hour', 'station'] if 'datetime_hour' in df.columns else ['date', 'hour']
df = df.sort_values(sort_cols).reset_index(drop=True)

feature_columns = [
    'station_id',
    'hour', 'lat', 'lng',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h'
]
target_column = 'net_demand'

missing = [c for c in feature_columns + [target_column] if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

X = df[feature_columns]
y = df[target_column].astype('float32')

if 'datetime_hour' in df.columns:
    cutoff_ts = df['datetime_hour'].max() - pd.Timedelta(days=7)
    train_mask = df['datetime_hour'] <= cutoff_ts
    test_mask = df['datetime_hour'] > cutoff_ts
    print('cutoff_ts:', cutoff_ts)
else:
    cutoff_date = df['date'].max() - pd.Timedelta(days=7)
    train_mask = df['date'] <= cutoff_date
    test_mask = df['date'] > cutoff_date
    print('cutoff_date:', cutoff_date)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print('train rows:', len(X_train))
print('test rows :', len(X_test))
print('test zero ratio:', float((y_test == 0).mean()))


In [ ]:
# Train or load RF artifacts
run_training = True

artifact_dir = os.path.join(project_folder, 'artifacts/model_training/random_forest/v2_0')
os.makedirs(artifact_dir, exist_ok=True)

model_path = os.path.join(artifact_dir, 'rf_model.joblib')
metrics_path = os.path.join(artifact_dir, 'metrics.json')
predictions_path = os.path.join(artifact_dir, 'test_predictions.parquet')
feature_importance_path = os.path.join(artifact_dir, 'feature_importance.csv')

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

if run_training:
    t0 = time.perf_counter()
    rf.fit(X_train, y_train)
    train_seconds = time.perf_counter() - t0
    joblib.dump(rf, model_path)
    print('Training complete, saved model:', model_path)
    model = rf
else:
    model = joblib.load(model_path)
    train_seconds = 0.0
    print('Loaded model:', model_path)


In [ ]:
# Evaluate model and save artifacts
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

train_rmse = float(np.sqrt(mean_squared_error(y_train, pred_train)))
train_mae = float(mean_absolute_error(y_train, pred_train))
test_rmse = float(np.sqrt(mean_squared_error(y_test, pred_test)))
test_mae = float(mean_absolute_error(y_test, pred_test))

baselines = {
    'zero': np.zeros(len(y_test), dtype=np.float32),
    'lag_1h': X_test['lag_1h'].to_numpy(),
    'lag_24h': X_test['lag_24h'].to_numpy(),
    'rolling_mean_3h': X_test['rolling_mean_3h'].to_numpy(),
}
baseline_rows = []
for name, bp in baselines.items():
    baseline_rows.append({
        'model': name,
        'rmse': float(np.sqrt(mean_squared_error(y_test, bp))),
        'mae': float(mean_absolute_error(y_test, bp)),
    })

perf_df = pd.DataFrame([
    {'split': 'train', 'rmse': train_rmse, 'mae': train_mae},
    {'split': 'test', 'rmse': test_rmse, 'mae': test_mae},
])

print('\n--- Training Time ---')
print(f'{train_seconds:.2f} sec ({train_seconds/60:.2f} min)')

print('\n--- Model Performance ---')
display(perf_df)

print('\n--- Test Baselines ---')
display(pd.DataFrame(baseline_rows))

pred_df = pd.DataFrame({
    'y_true': y_test.to_numpy(),
    'y_pred': pred_test,
    'lag_1h': X_test['lag_1h'].to_numpy(),
    'lag_24h': X_test['lag_24h'].to_numpy(),
    'rolling_mean_3h': X_test['rolling_mean_3h'].to_numpy(),
})
if 'datetime_hour' in df.columns:
    pred_df['datetime_hour'] = df.loc[test_mask, 'datetime_hour'].to_numpy()
if 'station' in df.columns:
    pred_df['station'] = df.loc[test_mask, 'station'].to_numpy()

pred_df.to_parquet(predictions_path, index=False)

metrics = {
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'train_seconds': train_seconds,
    'train_rmse': train_rmse,
    'train_mae': train_mae,
    'test_rmse': test_rmse,
    'test_mae': test_mae,
    'baseline_test': baseline_rows,
    'feature_columns': feature_columns,
    'model_path': model_path,
    'predictions_path': predictions_path,
}
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('\nSaved artifacts:')
print('-', model_path)
print('-', metrics_path)
print('-', predictions_path)


In [ ]:
# Plot diagnostics
sample_n = min(8000, len(y_test))
idx = np.random.default_rng(42).choice(len(y_test), size=sample_n, replace=False)

y_true_s = y_test.iloc[idx].to_numpy()
y_pred_s = pred_test[idx]

nz = y_test != 0
nz_idx = np.where(nz.to_numpy())[0]
if len(nz_idx) > 0:
    nz_n = min(8000, len(nz_idx))
    nz_sample = np.random.default_rng(42).choice(nz_idx, size=nz_n, replace=False)
    y_true_nz = y_test.iloc[nz_sample].to_numpy()
    y_pred_nz = pred_test[nz_sample]
else:
    y_true_nz = np.array([])
    y_pred_nz = np.array([])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].scatter(y_true_s, y_pred_s, alpha=0.25, s=16, color='#2563eb')
mn = float(min(y_true_s.min(), y_pred_s.min()))
mx = float(max(y_true_s.max(), y_pred_s.max()))
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=2)
axes[0].set_title('Actual vs Predicted (All Test Sample)')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].grid(alpha=0.3)

if len(y_true_nz) > 0:
    axes[1].scatter(y_true_nz, y_pred_nz, alpha=0.25, s=16, color='#059669')
    mn2 = float(min(y_true_nz.min(), y_pred_nz.min()))
    mx2 = float(max(y_true_nz.max(), y_pred_nz.max()))
    axes[1].plot([mn2, mx2], [mn2, mx2], 'r--', lw=2)
    axes[1].set_title('Actual vs Predicted (Non-zero Test Sample)')
    axes[1].set_xlabel('Actual')
    axes[1].set_ylabel('Predicted')
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No non-zero rows', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

residuals = y_test.to_numpy() - pred_test
plt.figure(figsize=(10, 5))
sns.histplot(residuals, bins=60, kde=True, color='#7c3aed')
plt.title('Residual Distribution (Test)')
plt.xlabel('Residual (y_true - y_pred)')
plt.tight_layout()
plt.show()


In [ ]:
# RF-specific artifact and plot: feature importance
fi = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)
fi.to_csv(feature_importance_path, index=False)
print('Saved feature importance:', feature_importance_path)

display(fi.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=fi.head(12), x='importance', y='feature', color='#2563eb')
plt.title('RandomForest Feature Importance')
plt.tight_layout()
plt.show()
